Tests data ingestion is the same as original code by comparing questionnaire, microdata and paradata output.

In [67]:
from pathlib import Path
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
from collections import Counter
import math
import json
from pandas.api import types as ptypes
import pyarrow.parquet as pq
import ast

from rissk.config import DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, INTERIM_DATA_DIR, PROJ_ROOT
from rissk.utils.testing_utils import compare_parquet_files

In [68]:
# SURVEY = "pmpmd"
# SURVEY = "hies2024"
SURVEY = "slchbs"

In [69]:
# This will remove the empty lists that were present in legacy data and are not present in the new Kedro outputs, 
# to ensure a more apples-to-apples comparison of the microdata. 
# It also handles stringified lists that may contain only-missing values (e.g., "[nan, nan]") by normalizing them to empty lists. 
# The drop_rows option allows for optionally removing rows where the specified columns contain empty lists after normalization, 
# which is relevant for the TextListQuestion rows in this test.

def clean_empty_lists(df: pd.DataFrame, columns: list = None, drop_rows: bool = False) -> pd.DataFrame:
    """Normalize list-like cells and optionally drop rows where the specified columns
    contain only-missing lists (e.g., [nan, nan] or stringified equivalents).

    Special handling for the token '##N/A##':
    - If a list mixes real-missing values (NaN) and the token '##N/A##', treat the list
      as empty (i.e., normalize to []).
    - If a list contains only the token '##N/A##' (no real-missing), leave it as-is.

    Args:
        df: input DataFrame
        columns: list of column names to clean
        drop_rows: if True, drop rows where any of the listed columns is an empty list
                   after normalization (i.e., [], or parsed [] from string like "[nan]").
    """
    if (columns is None) or (len(columns) == 0):
        print("No columns specified for cleaning empty lists. Returning original DataFrame.")
        return df

    def is_strict_missing(x):
        """True for NaN or common missing string tokens (excluding '##N/A##')."""
        try:
            if pd.isna(x):
                return True
        except Exception:
            pass
        if isinstance(x, str):
            t = x.strip().strip('\"\'')
            if t.lower() in ('nan', 'none', 'null', ''):
                return True
        return False

    def is_na_token(x):
        """True for the explicit token '##N/A##' (trim quotes and whitespace)."""
        if not isinstance(x, str):
            return False
        t = x.strip().strip('\"\'')
        return t == '##N/A##'

    def parse_if_list_str(x):
        # Already a Python list
        if isinstance(x, list):
            # Determine membership types
            if len(x) == 0:
                return []
            strict_missing_flags = [is_strict_missing(el) for el in x]
            na_token_flags = [is_na_token(el) for el in x]
            # If all elements are strict-missing -> empty
            if all(strict_missing_flags):
                return []
            # If all elements are either strict-missing or na-token, and at least one strict-missing -> empty
            if all(sm or nt for sm, nt in zip(strict_missing_flags, na_token_flags)) and any(strict_missing_flags):
                return []
            # Otherwise leave original list (including the case all are na-token)
            return x

        # Not a string -> nothing to do
        if not isinstance(x, str):
            return x
        s = x.strip()
        # Not a list-like string
        if not (s.startswith('[') and s.endswith(']')):
            return x
        # try json then ast
        try:
            val = json.loads(s)
        except Exception:
            try:
                val = ast.literal_eval(s)
            except Exception:
                val = None
        # If parsed to a Python list, evaluate missingness using same rules
        if isinstance(val, list):
            if len(val) == 0:
                return []
            strict_missing_flags = [is_strict_missing(el) for el in val]
            na_token_flags = [is_na_token(el) for el in val]
            if all(strict_missing_flags):
                return []
            if all(sm or nt for sm, nt in zip(strict_missing_flags, na_token_flags)) and any(strict_missing_flags):
                return []
            return val
        # Handle unquoted or mixed token lists like "[nan, '##N/A##']" by manual parse
        inner = s[1:-1].strip()
        if inner == "":
            return []
        parts = [p.strip() for p in inner.split(',')]
        if len(parts) > 0:
            strict_missing_flags = [is_strict_missing(p.strip().strip('\"\'')) for p in parts]
            na_token_flags = [is_na_token(p.strip().strip('\"\'')) for p in parts]
            if all(strict_missing_flags):
                return []
            if all(sm or nt for sm, nt in zip(strict_missing_flags, na_token_flags)) and any(strict_missing_flags):
                return []
        # otherwise leave original string (do not attempt risky eval)
        return x

    for col in columns:
        if col not in df.columns:
            continue
        # If real lists are present, normalize them (and clean all-missing lists)
        if df[col].apply(lambda x: isinstance(x, list)).any():
            def clean_list_cell(x):
                if isinstance(x, list):
                    if len(x) == 0:
                        return []
                    strict_missing_flags = [is_strict_missing(el) for el in x]
                    na_token_flags = [is_na_token(el) for el in x]
                    if all(strict_missing_flags):
                        return []
                    if all(sm or nt for sm, nt in zip(strict_missing_flags, na_token_flags)) and any(strict_missing_flags):
                        return []
                    return x
                if isinstance(x, str):
                    return parse_if_list_str(x)
                return x
            df[col] = df[col].apply(clean_list_cell)
        else:
            df[col] = df[col].apply(parse_if_list_str)

    if drop_rows:
        # Build mask for rows to drop: any specified column is an empty list
        drop_mask = pd.Series(False, index=df.index)
        for col in columns:
            if col not in df.columns:
                continue
            drop_mask = drop_mask | df[col].apply(lambda x: isinstance(x, list) and len(x) == 0)
        if drop_mask.any():
            df = df.loc[~drop_mask].reset_index(drop=True)

    return df

In [70]:
SURVEY = "pmpmd"
df_microdata = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))
df_microdata_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro","data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))


In [71]:
print(df_microdata['value'][236:237])

236    [nan, '##N/A##', nan, '##N/A##', nan, '##N/A##...
Name: value, dtype: object


In [72]:
# Normalize TextListQuestion `value` cells and drop rows with only-missing lists
mask_TextListQuestion = df_microdata['qtype'] == 'TextListQuestion'
print('TextListQuestion rows before:', mask_TextListQuestion.sum())
# Normalize list-like cells in the subset (do not drop rows in the subset call)
df_microdata.loc[mask_TextListQuestion, 'value'] = (
    clean_empty_lists(df_microdata.loc[mask_TextListQuestion].copy(), ['value'], drop_rows=False)['value']
)
# Drop rows where the 'value' column is an empty list for TextListQuestion rows
drop_mask = df_microdata['qtype'].eq('TextListQuestion') & df_microdata['value'].apply(lambda x: isinstance(x, list) and len(x) == 0)
print('Rows to drop (TextListQuestion with empty list):', drop_mask.sum())
if drop_mask.any():
    df_microdata = df_microdata.loc[~drop_mask].reset_index(drop=True)
    
print('TextListQuestion rows after:', (df_microdata['qtype'] == 'TextListQuestion').sum())

TextListQuestion rows before: 4672
Rows to drop (TextListQuestion with empty list): 3267
TextListQuestion rows after: 1405


In [73]:
def test_fast(SURVEY: str):
    # original files
    df_para = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "20_INTERIM", "paradata.parquet"))
    df_questionnaire = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "questionnaire.parquet"))
    df_microdata = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))
    try:
        df_para_processed = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "paradata.parquet"))
        # df_para_active = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "paradata_active.parquet"))
    except Exception as e:
        print(f"Error reading paradata_processed or paradata_active: {e}")
        df_para_processed = pd.DataFrame()
        # df_para_active = pd.DataFrame()

    # Normalize TextListQuestion `value` cells and drop rows with only-missing lists
    mask_TextListQuestion = df_microdata['qtype'] == 'TextListQuestion'
    print('TextListQuestion rows before:', mask_TextListQuestion.sum())
    # Normalize list-like cells in the subset (do not drop rows in the subset call)
    df_microdata.loc[mask_TextListQuestion, 'value'] = (
        clean_empty_lists(df_microdata.loc[mask_TextListQuestion].copy(), ['value'], drop_rows=False)['value']
    )
    # Drop rows where the 'value' column is an empty list for TextListQuestion rows
    drop_mask = df_microdata['qtype'].eq('TextListQuestion') & df_microdata['value'].apply(lambda x: isinstance(x, list) and len(x) == 0)
    print('Rows to drop (TextListQuestion with empty list):', drop_mask.sum())
    if drop_mask.any():
        df_microdata = df_microdata.loc[~drop_mask].reset_index(drop=True)
    print('TextListQuestion rows after:', (df_microdata['qtype'] == 'TextListQuestion').sum())

    # Kedro pipeline outputs
    df_para_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "20_INTERIM", "paradata.parquet"))
    df_questionnaire_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "questionnaire.parquet"))
    df_microdata_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))
    try:
        df_para_processed_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "paradata_processed.parquet"))
        # df_para_active_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "paradata_active.parquet"))
    except Exception as e:
        print(f"Error reading paradata_processed_kedro or paradata_active_kedro: {e}")
        df_para_processed_kedro = pd.DataFrame()
        # df_para_active_kedro = pd.DataFrame()

    for df_name, df_orig, df_kedro in [
        ("paradata", df_para, df_para_kedro),
        ("questionnaire", df_questionnaire, df_questionnaire_kedro),
        ("microdata", df_microdata, df_microdata_kedro),
        ("paradata_processed", df_para_processed, df_para_processed_kedro),
        # ("paradata_active", df_para_active, df_para_active_kedro)
    ]:
        try:
            print(30 * "=" + f" {df_name.upper()} " + 30 * "=")
            print('Shape:', f"Original - {df_orig.shape}, Kedro - {df_kedro.shape}")
            if df_name in ["questionnaire", "microdata"]:
                print('QNR Sequence:', f"Original - {df_orig['qnr_seq'].nunique()}, Kedro - {df_kedro['qnr_seq'].nunique()}")
            print('QNR Version:', f"Original - {df_orig['qnr_version'].unique()}, Kedro - {df_kedro['qnr_version'].unique()}")
            # print('QNR Version empty:', f"Original - {df_orig['qnr_version'].isna().sum()}, Kedro - {df_kedro['qnr_version'].isna().sum()}")
            print('QNR:', f"Original - {df_orig['qnr'].unique()}, Kedro - {df_kedro['qnr'].unique()}")
            if df_name == "microdata":
                # print('Values:', f"Original - {df_orig['value'].nunique()}, Kedro - {df_kedro['value'].nunique()}")
                print('interview__id:', f"Original - {df_orig['interview__id'].nunique()}, Kedro - {df_kedro['interview__id'].nunique()}")
                print('interview_id by qnr', f"Original - {df_orig.groupby(['qnr', 'qnr_version'])['interview__id'].nunique().to_dict()}, Kedro - {df_kedro.groupby(['qnr', 'qnr_version'])['interview__id'].nunique().to_dict()}")
            if df_name in ["paradata", "paradata_processed"]:
                print('interview__id:', f"Original - {df_orig['interview__id'].nunique()}, Kedro - {df_kedro['interview__id'].nunique()}")
        except Exception as e:
            print(f"Error comparing {df_name}: {e}")
        
    

In [74]:
def test_cell(SURVEY: str, survey_details: dict = None):
    if survey_details is None:
        survey_details = {}

    # original files
    df_para = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "20_INTERIM", "paradata.parquet"))
    df_para.sort_values(by=['qnr_version', 'qnr', 'interview__id', 'qnr_seq'], inplace=True, ignore_index=True)

    df_questionnaire = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "questionnaire.parquet"))
    df_questionnaire.sort_values(by=['qnr_version', 'qnr', 'qnr_seq'], inplace=True, ignore_index=True)
    print('questionnaire rows with categories', df_questionnaire['categories_id'].notna().sum())

    df_microdata = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))
    
    # Normalize TextListQuestion `value` cells and drop rows with only-missing lists
    mask_TextListQuestion = df_microdata['qtype'] == 'TextListQuestion'
    # print('TextListQuestion rows before:', mask_TextListQuestion.sum())
    # Normalize list-like cells in the subset (do not drop rows in the subset call)
    df_microdata.loc[mask_TextListQuestion, 'value'] = (
        clean_empty_lists(df_microdata.loc[mask_TextListQuestion].copy(), ['value'], drop_rows=False)['value']
    )
    # Drop rows where the 'value' column is an empty list for TextListQuestion rows
    drop_mask = df_microdata['qtype'].eq('TextListQuestion') & df_microdata['value'].apply(lambda x: isinstance(x, list) and len(x) == 0)
    print('Rows to drop (TextListQuestion with empty list):', drop_mask.sum())
    if drop_mask.any():
        df_microdata = df_microdata.loc[~drop_mask].reset_index(drop=True)
    # print('TextListQuestion rows after:', (df_microdata['qtype'] == 'TextListQuestion').sum())

    df_microdata.sort_values(by=['qnr_version', 'qnr', 'interview__id', 'qnr_seq'], inplace=True, ignore_index=True)
    
    try:
        df_para_processed = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "paradata.parquet"))
        df_para_processed.sort_values(by=['qnr_version', 'qnr', 'interview__id'], inplace=True, ignore_index=True)
        print('paradata_processed rows with categories', df_para_processed['categories'].notna().sum())
        # df_para_active = pd.read_parquet(PROJ_ROOT.joinpath("data", SURVEY, "latest", "30_PROCESSED", "paradata_active.parquet"))
        # df_para_active.sort_values(by=['qnr_version', 'qnr', 'interview__id'], inplace=True, ignore_index=True)
    except Exception as e:
        print(f"Error reading paradata_processed or paradata_active: {e}")
        df_para_processed = pd.DataFrame()
        # df_para_active = pd.DataFrame()

    # Kedro pipeline outputs
    df_para_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "20_INTERIM", "paradata.parquet"))
    df_para_kedro.sort_values(by=['qnr_version', 'qnr', 'interview__id', 'qnr_seq'], inplace=True, ignore_index=True)
    df_questionnaire_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "questionnaire.parquet"))
    df_questionnaire_kedro.sort_values(by=['qnr_version', 'qnr', 'qnr_seq'], inplace=True, ignore_index=True)
    df_microdata_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))
    df_microdata_kedro.sort_values(by=['qnr_version', 'qnr', 'interview__id', 'qnr_seq'], inplace=True, ignore_index=True)
    
    try:
        df_para_processed_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "paradata_processed.parquet"))
        df_para_processed_kedro.sort_values(by=['qnr_version', 'qnr', 'interview__id'], inplace=True, ignore_index=True)
        # df_para_active_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "paradata_active.parquet"))
        # df_para_active_kedro.sort_values(by=['qnr_version', 'qnr', 'interview__id'], inplace=True, ignore_index=True)
    except Exception as e:
        print(f"Error reading paradata_processed_kedro or paradata_active_kedro: {e}")
        df_para_processed_kedro = pd.DataFrame()
        # df_para_active_kedro = pd.DataFrame()

    for df_name, df_orig, df_kedro in [
        ("paradata", df_para, df_para_kedro),
        ("questionnaire", df_questionnaire, df_questionnaire_kedro),
        ("microdata", df_microdata, df_microdata_kedro),
        ("paradata_processed", df_para_processed, df_para_processed_kedro),
        # ("paradata_active", df_para_active, df_para_active_kedro)
    ]:
        try:
            same, details = compare_parquet_files(df_kedro, df_orig, check='cells')
            # store details per survey and per df_name (don't overwrite previous df_name entries)
            survey_details.setdefault(SURVEY, {})[df_name] = details
            print(30 * "=" + f" {df_name.upper()} CELL COMPARISON " + 30 * "=")
            print(same)
            print(details['shape'])
            print(details['columns'])
            print(details['dtypes'])
            print(details['cell_compare'])
            print('Number of cell differences:', details['cell_compare']['total_cell_differences'])
            try:
                print('Diff DF shape:', details["diff_df"].shape)
                # display(details["diff_df"])
            except Exception as e:
                print(f"Error displaying diff_df for {df_name}: {e}")
        except Exception as e:
            print(f"Error comparing {df_name}: {e}")
    return survey_details

In [ ]:
for SURVEY in ["pmpmd", "hies2024", "slchbs", "fbf house holduntitled folder"]:
    print((80 + len(f" TESTING SURVEY: {SURVEY.upper()} ")) * "=")
    print(40 * "=" + f" TESTING SURVEY: {SURVEY.upper()} " + 40 * "=")
    print((80 + len(f" TESTING SURVEY: {SURVEY.upper()} ")) * "=")
    test_fast(SURVEY)

======================================== TESTING SURVEY: PMPMD ========================================
TextListQuestion rows before: 4672
Rows to drop (TextListQuestion with empty list): 3267
TextListQuestion rows after: 1405


In [ ]:
survey_details = {}
for SURVEY in ["pmpmd", "hies2024", "slchbs", "fbf house holduntitled folder"]:
    print((80 + len(f" TESTING SURVEY: {SURVEY.upper()} ")) * "=")
    print(40 * "=" + f" TESTING SURVEY: {SURVEY.upper()} " + 40 * "=")
    print((80 + len(f" TESTING SURVEY: {SURVEY.upper()} ")) * "=")
    test_cell(SURVEY, survey_details)

survey_details

======================================== TESTING SURVEY: PMPMD ========================================
questionnaire rows with categories 641
Rows to drop (TextListQuestion with empty list): 3267
Error reading paradata_processed or paradata_active: 'categories'
============================== PARADATA CELL COMPARISON ==============================
False
{'equal': True, 'shape_a': (1766171, 27), 'shape_b': (1766171, 27)}
{'different_columns': [], 'equal': True, 'only_in_a': [], 'only_in_b': []}
{'mismatched_columns': [], 'equal': True}
{'checked': True, 'columns_with_differences': ['answer_sequence', 'n_answers'], 'total_cell_differences': 900828, 'rows_compared': 1766171, 'note': 'aligned by index intersection'}
Number of cell differences: 900828
Diff DF shape: (900828, 4)
============================== QUESTIONNAIRE CELL COMPARISON ==============================
False
{'equal': True, 'shape_a': (3000, 38), 'shape_b': (3000, 38)}
{'different_columns': [], 'equal': True, 'only_in_a': []

KeyboardInterrupt: 

In [ ]:
display(survey_details['hies2024']['microdata']['diff_df'])

In [ ]:
display(survey_details['pmpmd']['microdata']['diff_df'])

,index,column,value_a,value_b
0,0,answer_sequence,"[10140, 10142, 10147, 15218, 15220, 15233, 182...",nan
1,0,n_answers,18.0,__NA__
2,1,answer_sequence,"[3004257, 3004259, 3004553, 3004951, 3004953, ...",nan
3,1,n_answers,37.0,__NA__
4,7,value,"['Бат', 1, 'Цэцэг', 2]","[Бат, ##N/A##, Цэцэг, ##N/A##]"
...,...,...,...,...
6390225,271549,categories_id,__NA__,204f1d1c-5bae-414e-811d-fea87daf3712
6390226,271549,parents,V: RESULT,B: MEMBERS > MEMBER
6390227,271549,parent_1,V: RESULT,B: MEMBERS
6390228,271549,parent_2,__NA__,MEMBER


In [ ]:
def summary_table_for_survey(SURVEY: str, datasets=None):
    """Produce a summary DataFrame with requested comparison stats for a survey.

    Columns: SURVEY, df, shape_equal, shape_rows, shape_cols, dtype_equal,
    different_columns, columns_with_differences, num_cell_differences, questionnaire_categories_count
    """
    if datasets is None:
        datasets = ['paradata', 'questionnaire', 'microdata', 'paradata_processed']
    rows = []
    for df_name in datasets:
        try:
            orig_path = PROJ_ROOT.joinpath('data', SURVEY, 'latest')
            kedro_path = PROJ_ROOT.joinpath('rissk_kedro', 'data', SURVEY, 'latest')
            if df_name == 'paradata':
                df_orig = pd.read_parquet(orig_path.joinpath('20_INTERIM', 'paradata.parquet'))
                df_kedro = pd.read_parquet(kedro_path.joinpath('20_INTERIM', 'paradata.parquet'))
            else:
                df_orig = pd.read_parquet(orig_path.joinpath('30_PROCESSED', f'{df_name}.parquet'))
                df_kedro = pd.read_parquet(kedro_path.joinpath('30_PROCESSED', f'{df_name}.parquet'))

            same, details = compare_parquet_files(df_kedro, df_orig, check='cells')

            shape_equal = bool(details.get('shape', {}).get('equal', False))
            shape_a = details.get('shape', {}).get('shape_a', (None, None))
            try:
                shape_rows = int(shape_a[0])
            except Exception:
                shape_rows = None
            try:
                shape_cols = int(shape_a[1])
            except Exception:
                shape_cols = None

            dtype_equal = bool(details.get('dtypes', {}).get('equal', False))
            different_columns = details.get('columns', {}).get('different_columns', [])
            cols_with_diff = details.get('cell_compare', {}).get('columns_with_differences', [])
            num_cell_diffs = int(details.get('cell_compare', {}).get('total_cell_differences', 0))

            questionnaire_categories_count = None
            if df_name == 'questionnaire':
                try:
                    questionnaire_categories_count = int(df_orig['categories_id'].notna().sum())
                except Exception:
                    questionnaire_categories_count = None

            rows.append({
                'SURVEY': SURVEY,
                'df': df_name,
                'shape_bool': shape_equal,
                'shape[0]': shape_rows,
                'shape[1]': shape_cols,
                'dtype_bool': dtype_equal,
                'different_columns': ','.join(map(str, different_columns)) if different_columns else '',
                'columns_with_differences': ','.join(map(str, cols_with_diff)) if cols_with_diff else '',
                'Number of cell differences': num_cell_diffs,
                'questionnaire_categories_count': questionnaire_categories_count,
            })
        except Exception as e:
            rows.append({
                'SURVEY': SURVEY,
                'df': df_name,
                'shape_bool': False,
                'shape[0]': None,
                'shape[1]': None,
                'dtype_bool': False,
                'different_columns': '',
                'columns_with_differences': '',
                'Number of cell differences': None,
                'questionnaire_categories_count': None,
                'error': str(e)
            })
    return pd.DataFrame(rows)

# Example: produce and display table for a survey
# summary_df = summary_table_for_survey('hies2024')
# display(summary_df)
# print('summary_table_for_survey defined — call it with a survey name to produce the table.')

In [ ]:
# Run summaries for the four surveys, append into one table, and export as CSV
surveys = ["pmpmd", "hies2024", "slchbs", "fbf house holduntitled folder"]
all_dfs = []
for s in surveys:
    try:
        print(f'Generating summary for {s}...')
        df = summary_table_for_survey(s)
        # add survey column already present, ensure consistent order
        all_dfs.append(df)
    except Exception as e:
        print(f'Error for {s}: {e}')

if len(all_dfs) > 0:
    summary_all = pd.concat(all_dfs, ignore_index=True)
    out_dir = PROJ_ROOT.joinpath('data','reports')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_dir.joinpath('comparison_summary_all_surveys.csv')
    summary_all.to_csv(out_csv, index=False)
    print(f'Wrote summary CSV to: {out_csv}')
    display(summary_all)
else:
    print('No summaries were produced.')